# Solid-Link (slink) — Colab GPU 에이전트

이 노트북을 **순서대로 실행**하면 SOLID VM / VS Code에서 이 Colab의 GPU를 사용할 수 있습니다.

**사전 준비**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) 에서 무료 인증 토큰 발급
3. 아래 Cell 2의 `NGROK_AUTHTOKEN`과 `RELAY_SERVER_URL`을 채워주세요

> SSH 터널 대신 **HTTP 터널**을 사용하므로 ngrok 카드 등록 불필요

In [ ]:
# [1단계] 의존성 설치
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-300:] if r.stdout else '')
        print(r.stderr[-300:] if r.stderr else '')
        raise RuntimeError(f'명령어 실패: {" ".join(cmd)}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'pyngrok', 'jupyterlab'])
print('설치 완료')

In [ ]:
# [2단계] 설정값 입력 (반드시 채워주세요)
NGROK_AUTHTOKEN = ""  # https://dashboard.ngrok.com/get-started/your-authtoken
RELAY_SERVER_URL = ""  # 예: https://slink-relay.railway.app

if not NGROK_AUTHTOKEN or not RELAY_SERVER_URL:
    raise ValueError('NGROK_AUTHTOKEN 과 RELAY_SERVER_URL 을 모두 입력해주세요.')

In [ ]:
WARNING:pyngrok.process.ngrok:t=2026-05-04T11:24:03+0000 lvl=warn msg="failed to start tunnel" pg=/api/tunnels id=17fa9aff7eaeda4d err="failed to start tunnel: The endpoint 'https://unvented-decimal-endorphin.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nRR_NGROK_334\r\n"
---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
/usr/local/lib/python3.12/dist-packages/pyngrok/ngrok.py in api_request(url, method, data, params, timeout, auth)
    631     try:
--> 632         response = urlopen(request, encoded_data, timeout)
    633         response_data = response.read().decode("utf-8")

8 frames
HTTPError: HTTP Error 502: Bad Gateway

During handling of the above exception, another exception occurred:

PyngrokNgrokHTTPError                     Traceback (most recent call last)
/usr/local/lib/python3.12/dist-packages/pyngrok/ngrok.py in api_request(url, method, data, params, timeout, auth)
    651         logger.debug(f"Response {status_code}: {response_data.strip()}")
    652
--> 653         raise PyngrokNgrokHTTPError(f"ngrok client exception, API returned {status_code}: {response_data}",
    654                                     e.url,
    655                                     status_code, e.reason, e.headers, response_data)

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://unvented-decimal-endorphin.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [ ]:
# [6단계] 연결 유지 (세션이 끝날 때까지 실행)
import time, requests

print('연결 유지 중... (정지 버튼으로 세션 종료)')
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    try:
        requests.delete(f'{RELAY_SERVER_URL}/api/session/{code}', timeout=5)
    except Exception:
        pass
    print('세션을 종료했습니다.')